# Umubyeyi - Model 3: AfroXLMR fine-tune (Colab GPU)

Fine-tunes **AfroXLMR-base** (pretrained on Kinyarwanda) to classify Kinyarwanda text into the 6 maternal-wellness intents - the Kinyarwanda-direct model, no translation hop.

## How to run (≈15 min)
1. **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**
2. **Runtime → Run all**
3. The metrics print at the end, save to `kinyarwanda_finetune_metrics.json`, and auto-download.

Data is pulled directly from the GitHub repo, so there is nothing to upload.

In [ ]:
!pip install -q -U transformers datasets accelerate scikit-learn

In [ ]:
import torch, numpy as np, pandas as pd, json
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

RNG = 42; torch.manual_seed(RNG); np.random.seed(RNG)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'NO GPU - set Runtime to T4!')

MODEL = 'Davlan/afro-xlmr-base'
MAXLEN, BATCH, EPOCHS, LR = 128, 16, 5, 2e-5

URL = 'https://raw.githubusercontent.com/IrutingaboRaissa/UMUBYEYI/main/notebooks/amod_kinyarwanda.csv'
df = pd.read_csv(URL).dropna(subset=['context_rw','intent'])
labels = sorted(df['intent'].unique()); l2i = {l:i for i,l in enumerate(labels)}
df['y'] = df['intent'].map(l2i)
print('rows:', len(df), '| classes:', len(labels))

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(df['context_rw'].tolist(), df['y'].tolist(),
                                      test_size=0.20, random_state=RNG, stratify=df['y'].tolist())
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels)).to(DEVICE)

class DS(Dataset):
    def __init__(self, texts, ys):
        self.enc = tok(texts, truncation=True, padding='max_length', max_length=MAXLEN, return_tensors='pt')
        self.y = torch.tensor(ys)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return {k: v[i] for k, v in self.enc.items()} | {'labels': self.y[i]}

# class weights (handles imbalance)
counts = np.bincount(ytr, minlength=len(labels))
w = torch.tensor(counts.sum() / (len(labels) * counts), dtype=torch.float).to(DEVICE)
lossf = torch.nn.CrossEntropyLoss(weight=w)

tr = DataLoader(DS(Xtr, ytr), batch_size=BATCH, shuffle=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, int(0.1*len(tr)*EPOCHS), len(tr)*EPOCHS)

for ep in range(EPOCHS):
    model.train(); run = 0.0
    for batch in tr:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        opt.zero_grad()
        out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        loss = lossf(out.logits, batch['labels'])
        loss.backward(); opt.step(); sched.step(); run += loss.item()
    print(f'epoch {ep+1}/{EPOCHS}  avg_loss={run/len(tr):.3f}')

In [ ]:
model.eval(); preds = []
te = DataLoader(DS(Xte, yte), batch_size=32)
with torch.no_grad():
    for batch in te:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).logits
        preds.extend(logits.argmax(-1).cpu().tolist())

acc = accuracy_score(yte, preds)
pr, rc, f1, _ = precision_recall_fscore_support(yte, preds, average='macro', zero_division=0)
print(f'AfroXLMR (Kinyarwanda-direct):  acc={acc:.3f}  F1={f1:.3f}\n')
print(classification_report(yte, preds, target_names=labels, zero_division=0))

out = {'Model 3: AfroXLMR fine-tune (Kinyarwanda)': {
    'accuracy': round(float(acc),4), 'precision_macro': round(float(pr),4),
    'recall_macro': round(float(rc),4), 'f1_macro': round(float(f1),4),
    '_model': MODEL, '_epochs': EPOCHS, '_device': DEVICE}}
with open('kinyarwanda_finetune_metrics.json','w') as f: json.dump(out, f, indent=2)
print('\nSaved -> kinyarwanda_finetune_metrics.json')
try:
    from google.colab import files; files.download('kinyarwanda_finetune_metrics.json')
except Exception as e:
    print('(download skipped:', e, ')')